In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 📌 1. Introduction

Feature Engineering is the process of transforming raw data into meaningful features that improve model performance.

# 📥 2. Import Libraries

In [3]:
import numpy as np
import pandas as pd

# 📂 3. Load Dataset

In [4]:
df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 🧹 4. Basic Cleaning

In [6]:
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"], errors="ignore")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


# ⚙️ 5. Feature Engineering (CORE 🔥)

# 🔢 5.1 Family Size

In [7]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
#Insight: Larger families may have different survival chances
df[["SibSp", "Parch", "FamilySize"]].head()

,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


# 🧍 5.2 Is Alone

In [8]:
df["IsAlone"] = np.where(df["FamilySize"] == 1, 1, 0)

df[["FamilySize", "IsAlone"]].head()

,FamilySize,IsAlone
0,2,0
1,2,0
2,1,1
3,2,0
4,1,1


# 🎂 5.3 Age Groups (Binning)

In [9]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0,12,18,40,60,100],
    labels=["Child","Teen","Adult","Mid","Senior"]
)

df[["Age", "AgeGroup"]].head()

,Age,AgeGroup
0,22.0,Adult
1,38.0,Adult
2,26.0,Adult
3,35.0,Adult
4,35.0,Adult


# 💰 5.4 Fare Transformation (Log)

In [10]:
df["Fare_log"] = np.log1p(df["Fare"])

df[["Fare", "Fare_log"]].head()

,Fare,Fare_log
0,7.2500,2.110213
1,71.2833,4.280593
2,7.9250,2.188856
3,53.1000,3.990834
4,8.0500,2.202765


# 🏷️ 5.5 Fare Categories

In [11]:
df["FareCategory"] = pd.cut(
    df["Fare"],
    bins=[0,10,50,df["Fare"].max()],
    labels=["Low","Medium","High"]
)

df[["Fare", "FareCategory"]].head()

,Fare,FareCategory
0,7.2500,Low
1,71.2833,High
2,7.9250,Low
3,53.1000,High
4,8.0500,Low


# 🧾 5.6 Title Extraction (Advanced 🔥)

In [12]:
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

df[["Name", "Title"]].head()

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


# 🔢 5.7 Encoding (Convert to Numbers)

In [13]:
df = pd.get_dummies(df, columns=["Sex", "Embarked", "Title"], drop_first=True)

print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1      0   
2                             Heikkinen, Miss. Laina  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1      0   
4                           Allen, Mr. William Henry  35.0      0      0   

             Ticket     Fare  FamilySize  ...  Title_Major Title_Master  \
0         A/5 21171   7.2500           2  ...        False        False   
1          PC 17599  71.2833           2  ...        False        False   
2  STON/O2. 3101282   7.9250           1  ...        False        False   
3            113803  53.10

# 📊 6. Verify Features

In [14]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 33 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   PassengerId     891 non-null    int64   
 1   Survived        891 non-null    int64   
 2   Pclass          891 non-null    int64   
 3   Name            891 non-null    object  
 4   Age             891 non-null    float64 
 5   SibSp           891 non-null    int64   
 6   Parch           891 non-null    int64   
 7   Ticket          891 non-null    object  
 8   Fare            891 non-null    float64 
 9   FamilySize      891 non-null    int64   
 10  IsAlone         891 non-null    int64   
 11  AgeGroup        891 non-null    category
 12  Fare_log        891 non-null    float64 
 13  FareCategory    876 non-null    category
 14  Sex_male        891 non-null    bool    
 15  Embarked_Q      891 non-null    bool    
 16  Embarked_S      891 non-null    bool    
 17  Title_Col       

## 🧠 Key Insights

- 📌 **Family Size Impact**  
  Passengers traveling with family showed different survival patterns compared to those traveling alone. Smaller families and individuals had varying survival probabilities.

- 📌 **Title Extraction (Social Status)**  
  Extracted titles such as *Mr, Mrs, Miss* provide valuable information about a passenger’s social status, age group, and gender, which significantly influence survival chances.

- 📌 **Fare Transformation**  
  The Fare feature was highly skewed. Applying a log transformation helped normalize the distribution, making it more suitable for machine learning models.

- 📌 **Feature Encoding**  
  Categorical variables like *Sex, Embarked,* and *Title* were converted into numerical format using encoding techniques, enabling them to be used effectively in machine learning algorithms.

- 📌 **Overall Impact**  
  Feature engineering enhanced the dataset by creating more meaningful and informative variables, improving the model's ability to learn patterns.

## 🏁 Conclusion

In this notebook, we transformed raw data into meaningful features using various feature engineering techniques.

- Cleaned and prepared the dataset for analysis  
- Created new features such as FamilySize and IsAlone  
- Extracted hidden information like passenger titles  
- Applied transformations to improve data distribution  
- Converted categorical data into machine-readable format  

These steps play a crucial role in improving the performance of machine learning models.  

👉 Feature engineering is one of the most important stages in the data science pipeline, as better features lead to better predictions.